# Dataset Analysis

Explore the PII masking dataset used by the pipeline: split sizes, text lengths, entity counts, label distribution, and annotation sanity checks.

In [ ]:
import ast
import numbers
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from IPython.display import display

from src.pipeline.Evaluator import DEFAULT_LABEL_TO_PRESIDIO

## Configuration

In [ ]:
dataset_name = "NAMANDREWLV/pii-masking-95k-preencoded"

# Choose "all", "train", "validation", "val", or "test".
split_choice = "all"

# Set to an integer for faster profiling, or None to load all rows from selected splits.
sample_size_per_split = None

random_state = 42

## Load Dataset

In [ ]:
print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name)
available_splits = list(dataset.keys())
print("Available splits:", available_splits)

if split_choice == "all":
    selected_splits = available_splits
else:
    mapped_split = "validation" if split_choice == "val" and "validation" in dataset else split_choice
    if mapped_split not in dataset:
        raise ValueError(f"Split '{split_choice}' is not available. Available splits: {available_splits}")
    selected_splits = [mapped_split]

dfs = []
for split in selected_splits:
    split_df = dataset[split].to_pandas()
    if sample_size_per_split is not None and len(split_df) > sample_size_per_split:
        split_df = split_df.sample(n=sample_size_per_split, random_state=random_state)
    split_df["split"] = split
    dfs.append(split_df)

df = pd.concat(dfs, ignore_index=True)
if "text" in df.columns and "source_text" not in df.columns:
    df["source_text"] = df["text"]

print(f"Loaded {len(df):,} rows from splits: {selected_splits}")
display(df.head())
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "missing_pct": df.isna().mean().round(4)}))

## Split Distribution

In [ ]:
split_counts = df["split"].value_counts().rename_axis("split").reset_index(name="rows")
split_counts["pct"] = split_counts["rows"] / split_counts["rows"].sum()
display(split_counts)

ax = split_counts.plot.bar(x="split", y="rows", legend=False, figsize=(7, 4), color="#3b82f6")
ax.set_title("Rows by Split")
ax.set_xlabel("Split")
ax.set_ylabel("Rows")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Text Lengths

In [ ]:
df["char_len"] = df["source_text"].fillna("").str.len()
df["word_len"] = df["source_text"].fillna("").str.split().str.len()

length_summary = df.groupby("split")[["char_len", "word_len"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
display(length_summary)

overall_length_summary = df[["char_len", "word_len"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
display(overall_length_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["char_len"].hist(bins=60, ax=axes[0], color="#0f766e")
axes[0].set_title("Character Length Distribution")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Rows")

df["word_len"].hist(bins=60, ax=axes[1], color="#7c3aed")
axes[1].set_title("Word Length Distribution")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Rows")

plt.tight_layout()
plt.show()

## Normalize Entity Annotations

In [ ]:
def normalize_privacy_mask(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (SyntaxError, ValueError):
            return []
    if hasattr(value, "tolist"):
        value = value.tolist()
    if isinstance(value, dict):
        return [value]
    if isinstance(value, (list, tuple)):
        return [item for item in value if isinstance(item, dict)]
    return []


df["privacy_mask_norm"] = df["privacy_mask"].apply(normalize_privacy_mask)
df["entity_count"] = df["privacy_mask_norm"].str.len()
df["has_entity"] = df["entity_count"] > 0

entity_rows = []
for row_index, row in df.iterrows():
    text = row["source_text"] or ""
    for entity_index, entity in enumerate(row["privacy_mask_norm"]):
        start = entity.get("start")
        end = entity.get("end")
        label = entity.get("label")
        entity_rows.append(
            {
                "row_index": row_index,
                "entity_index": entity_index,
                "split": row["split"],
                "label": label,
                "mapped_label": DEFAULT_LABEL_TO_PRESIDIO.get(label),
                "start": start,
                "end": end,
                "span_len": end - start if isinstance(start, numbers.Integral) and isinstance(end, numbers.Integral) else None,
                "span_text": text[start:end] if isinstance(start, numbers.Integral) and isinstance(end, numbers.Integral) else None,
            }
        )

entities_df = pd.DataFrame(entity_rows)
print(f"Exploded {len(entities_df):,} entity annotations from {len(df):,} rows.")
display(df[["split", "source_text", "entity_count"]].head())
display(entities_df.head())

## Entity Counts

In [ ]:
entity_count_summary = df.groupby("split")["entity_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
display(entity_count_summary)

rows_with_entities = df.groupby("split")["has_entity"].agg(rows="count", rows_with_entities="sum")
rows_with_entities["rows_without_entities"] = rows_with_entities["rows"] - rows_with_entities["rows_with_entities"]
rows_with_entities["pct_with_entities"] = rows_with_entities["rows_with_entities"] / rows_with_entities["rows"]
display(rows_with_entities.reset_index())

ax = df["entity_count"].hist(bins=range(0, int(df["entity_count"].max()) + 2), figsize=(8, 4), color="#2563eb")
ax.set_title("Entities per Row")
ax.set_xlabel("Entity Count")
ax.set_ylabel("Rows")
plt.tight_layout()
plt.show()

## Label Distribution

In [ ]:
if entities_df.empty:
    print("No entity annotations found.")
else:
    label_counts = entities_df["label"].value_counts().rename_axis("label").reset_index(name="count")
    label_counts["pct"] = label_counts["count"] / label_counts["count"].sum()
    display(label_counts)

    mapped_counts = entities_df["mapped_label"].fillna("UNMAPPED").value_counts().rename_axis("mapped_label").reset_index(name="count")
    mapped_counts["pct"] = mapped_counts["count"] / mapped_counts["count"].sum()
    display(mapped_counts)

    top_n = min(25, len(label_counts))
    ax = label_counts.head(top_n).sort_values("count").plot.barh(x="label", y="count", legend=False, figsize=(9, 7), color="#0891b2")
    ax.set_title(f"Top {top_n} Entity Labels")
    ax.set_xlabel("Annotations")
    ax.set_ylabel("Label")
    plt.tight_layout()
    plt.show()

    ax = mapped_counts.sort_values("count").plot.barh(x="mapped_label", y="count", legend=False, figsize=(8, 4), color="#9333ea")
    ax.set_title("Mapped Presidio Entity Groups")
    ax.set_xlabel("Annotations")
    ax.set_ylabel("Mapped Label")
    plt.tight_layout()
    plt.show()

## Label Distribution by Split

In [ ]:
if entities_df.empty:
    print("No entity annotations found.")
else:
    label_by_split = pd.crosstab(entities_df["label"], entities_df["split"])
    label_by_split["total"] = label_by_split.sum(axis=1)
    label_by_split = label_by_split.sort_values("total", ascending=False)
    display(label_by_split)

    mapped_by_split = pd.crosstab(entities_df["mapped_label"].fillna("UNMAPPED"), entities_df["split"])
    mapped_by_split["total"] = mapped_by_split.sum(axis=1)
    mapped_by_split = mapped_by_split.sort_values("total", ascending=False)
    display(mapped_by_split)

## Span Lengths

In [ ]:
if entities_df.empty:
    print("No entity annotations found.")
else:
    span_summary = entities_df.groupby("label")["span_len"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).sort_values("count", ascending=False)
    display(span_summary)

    ax = entities_df["span_len"].dropna().hist(bins=60, figsize=(8, 4), color="#ea580c")
    ax.set_title("Entity Span Length Distribution")
    ax.set_xlabel("Characters")
    ax.set_ylabel("Annotations")
    plt.tight_layout()
    plt.show()

## Sanity Checks

In [ ]:
def is_valid_span(row):
    start = row["start"]
    end = row["end"]
    if not isinstance(start, numbers.Integral) or not isinstance(end, numbers.Integral):
        return False
    text_len = len(df.loc[row["row_index"], "source_text"] or "")
    return 0 <= start < end <= text_len


if entities_df.empty:
    print("No entity annotations found.")
else:
    entities_df["valid_span"] = entities_df.apply(is_valid_span, axis=1)
    entities_df["is_mapped"] = entities_df["mapped_label"].notna()

    sanity_summary = pd.DataFrame(
        [
            {"check": "rows", "count": len(df)},
            {"check": "entities", "count": len(entities_df)},
            {"check": "invalid_spans", "count": int((~entities_df["valid_span"]).sum())},
            {"check": "unmapped_labels", "count": int((~entities_df["is_mapped"]).sum())},
            {"check": "empty_text_rows", "count": int((df["source_text"].fillna("").str.len() == 0).sum())},
        ]
    )
    display(sanity_summary)

    unmapped_labels = sorted(entities_df.loc[~entities_df["is_mapped"], "label"].dropna().unique())
    print("Unmapped labels:", unmapped_labels)

    invalid_spans = entities_df.loc[~entities_df["valid_span"]].head(20)
    display(invalid_spans)

## Examples by Label

In [ ]:
examples_per_label = 5

if entities_df.empty:
    print("No entity annotations found.")
else:
    examples = (
        entities_df.dropna(subset=["label", "span_text"])
        .groupby("label", group_keys=False)
        .head(examples_per_label)
        [["label", "mapped_label", "split", "span_text", "span_len", "row_index"]]
        .sort_values(["label", "row_index"])
    )
    display(examples)